In [ ]:
# ==========================================
# CELL 1: SETUP & EXACT PATH RESOLUTION
# ==========================================
import os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision

try:
    from ultralytics import YOLO
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)
    from ultralytics import YOLO

# 1. Hardcoded paths based on your specific Kaggle environment
INPUT_ROOT = Path("/kaggle/input/competitions/road-vision/RoadVision_DUET")

# We use rglob for the CSV in case it is sitting in the root or a subfolder
TRAIN_CSV = next(iter(INPUT_ROOT.rglob("train.csv")), None)

# Your exact provided paths
SAMPLE_SUB = INPUT_ROOT / "test" / "sample_submission.csv"
TRAIN_IMAGES = INPUT_ROOT / "train" / "images"
TEST_IMAGES  = INPUT_ROOT / "test" / "images"

print(f"Train CSV path resolved: {TRAIN_CSV}")
print(f"Train Images path resolved: {TRAIN_IMAGES}")
print(f"Test Images path resolved: {TEST_IMAGES}")

assert TRAIN_CSV and TRAIN_IMAGES.exists() and TEST_IMAGES.exists(), "Path mapping failed! Check directory structure."

In [ ]:
# ==========================================
# CELL 2: K-FOLD CONFIGURATION
# ==========================================
K_FOLDS = 5
EPOCHS_PER_FOLD = 40  # Reduced slightly from 60 to ensure all 5 folds finish within Kaggle's 9-hour limit
MODEL_ARCH = "yolov8s.pt" # Or yolo11s.pt depending on your specific Ultralytics version
IMGSZ = 960
BATCH = 16
DEVICE = 0

# Inference settings
PRED_CONF = 0.001 
PRED_IOU = 0.60
ENSEMBLE_IOU = 0.65 # The IoU threshold for the final voting mechanism

BASE_TEMP_DIR = Path("/kaggle/temp/roadvision_kfold")
RUNS_DIR = Path("/kaggle/working/runs")

In [ ]:
# ==========================================
# CELL 3: TEMPORAL CHUNK SPLITTING & COMPILATION
# ==========================================
df = pd.read_csv(TRAIN_CSV)
df['stem'] = df['image_id'].str.replace('.txt', '', regex=False)

# Index images
train_idx = {p.stem: p for p in TRAIN_IMAGES.rglob("*.*") if p.suffix.lower() in ['.jpg', '.png', '.jpeg']}
df = df[df['stem'].isin(train_idx)].reset_index(drop=True)
stems = sorted(df['stem'].unique())

# Group by Camera Clip
by_clip = {}
for s in stems:
    clip_id = s.split('_')[0]
    by_clip.setdefault(clip_id, []).append(s)

# Create K Folds via continuous chunking
folds_val_stems = {i: [] for i in range(K_FOLDS)}

for clip, clip_stems in by_clip.items():
    clip_stems.sort() # Ensure chronological order
    # Split this clip into K roughly equal continuous chunks
    chunks = np.array_split(clip_stems, K_FOLDS)
    for i in range(K_FOLDS):
        folds_val_stems[i].extend(chunks[i])

print(f"Mathematical Split Complete. Total unique images: {len(stems)}")
for i in range(K_FOLDS):
    print(f"Fold {i} Validation Images: {len(folds_val_stems[i])}")

# Group annotations
groups = {k: g for k, g in df.groupby("stem")}

def compile_fold(fold_idx):
    """Compiles the YOLO directory matrix for a specific fold."""
    fold_dir = BASE_TEMP_DIR / f"fold_{fold_idx}"
    if fold_dir.exists(): shutil.rmtree(fold_dir)
    
    for split in ["train", "val"]:
        (fold_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (fold_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        
    val_set = set(folds_val_stems[fold_idx])
    
    for stem in stems:
        split = "val" if stem in val_set else "train"
        src = train_idx[stem]
        dst_img = fold_dir / "images" / split / src.name
        
        # O(1) Symlinking
        os.symlink(src.resolve(), dst_img)
        
        # Write annotation tensor
        g = groups[stem]
        lines = [f"{int(r['class_id'])} {r['x_center']:.6f} {r['y_center']:.6f} {r['width']:.6f} {r['height']:.6f}"
                 for _, r in g.iterrows()]
        (fold_dir / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))
        
    # Generate YAML
    yaml_text = f"path: {fold_dir}\ntrain: images/train\nval: images/val\nnc: 13\nnames: {[str(i) for i in range(13)]}\n"
    (fold_dir / "data.yaml").write_text(yaml_text)
    return str(fold_dir / "data.yaml")

# Compile all folds to disk instantly
fold_yamls = [compile_fold(i) for i in range(K_FOLDS)]
print("All K-Fold directories compiled via symlinks.")

In [ ]:
# ==========================================
# CELL 4: SEQUENTIAL TRAINING LOOP
# ==========================================
print("Initiating K-Fold Training Matrix...")

for fold in range(K_FOLDS):
    print(f"\n{'='*40}\nCOMMENCING FOLD {fold}\n{'='*40}")
    model = YOLO(MODEL_ARCH)
    
    model.train(
        data=fold_yamls[fold],
        epochs=EPOCHS_PER_FOLD,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        project=RUNS_DIR,
        name=f"fold_{fold}",
        mosaic=1.0,
        close_mosaic=10,
        verbose=False # Keep logs clean
    )
    print(f"Fold {fold} complete. Weights mapped to: {RUNS_DIR}/fold_{fold}/weights/best.pt")

In [ ]:
# ==========================================
# CELL 5: ENSEMBLE INFERENCE & SUBMISSION  (CORRECTED)
#   - builds a row for EVERY test image (sample_submission only has 10 rows!)
#   - image_id = stem with NO extension
#   - Weighted Box Fusion instead of NMS (averages agreeing boxes -> better mAP)
# ==========================================
try:
    from ensemble_boxes import weighted_boxes_fusion
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "ensemble-boxes"], check=True)
    from ensemble_boxes import weighted_boxes_fusion

# Enumerate the real test set (NOT sample_submission, which is only a 10-row format sample)
test_idx = {p.stem: p for p in TEST_IMAGES.rglob("*.*")
            if p.suffix.lower() in [".jpg", ".png", ".jpeg"]}
test_stems = sorted(test_idx.keys())
print(f"Test images to predict: {len(test_stems)}  (expect 327)")

print("Loading the K-Fold ensemble...")
models = [YOLO(f"{RUNS_DIR}/fold_{i}/weights/best.pt") for i in range(K_FOLDS)]

WBF_IOU = 0.6
SKIP_THR = 0.0001   # low threshold keeps low-confidence boxes -> better mAP recall

rows = []
for stem in test_stems:
    p = str(test_idx[stem])
    boxes_l, scores_l, labels_l = [], [], []
    for m in models:
        b = m.predict(source=p, imgsz=IMGSZ, conf=0.001, iou=PRED_IOU,
                      max_det=300, verbose=False)[0].boxes
        if b is not None and len(b) > 0:
            boxes_l.append(b.xyxyn.cpu().numpy().tolist())     # normalized [x1,y1,x2,y2]
            scores_l.append(b.conf.cpu().numpy().tolist())
            labels_l.append(b.cls.cpu().numpy().astype(int).tolist())

    if not boxes_l:
        rows.append({"image_id": stem, "PredictionString": ""})
        continue

    fb, fs, fl = weighted_boxes_fusion(boxes_l, scores_l, labels_l,
                                       iou_thr=WBF_IOU, skip_box_thr=SKIP_THR)
    parts = []
    for i in np.argsort(fs)[::-1]:                              # confidence-descending
        x1, y1, x2, y2 = fb[i]
        parts += [str(int(fl[i])), f"{fs[i] * 100:.2f}",
                  f"{(x1 + x2) / 2:.6f}", f"{(y1 + y2) / 2:.6f}",
                  f"{x2 - x1:.6f}", f"{y2 - y1:.6f}"]
    rows.append({"image_id": stem, "PredictionString": " ".join(parts)})

sub = pd.DataFrame(rows)                # image_id = stem, NO extension (sample_submission's .txt is wrong)
output_csv = "/kaggle/working/submission.csv"
sub.to_csv(output_csv, index=False)

n_with = int((sub["PredictionString"].str.len() > 0).sum())
print(f"\nSaved {output_csv}  | rows: {len(sub)} (expect 327)  | images with >=1 detection: {n_with}")
print(sub.head())
assert len(sub) == len(test_stems), "Row count mismatch - submission is incomplete!"
